# ETL — Observatório do Ecossistema Industrial e de Serviços de São José dos Campos

**Projeto:** API — Fatec São José dos Campos · Curso de Logística
**Cliente:** CADI / Secretaria de Desenvolvimento Econômico de SJC
**Fonte de dados:** RAIS_ESTAB_PUB (Ministério do Trabalho e Emprego) — base nacional de estabelecimentos

Este notebook implementa o pipeline **ETL (Extract, Transform, Load)** que:

1. **Extract** — lê o arquivo bruto `RAIS_ESTAB_PUB.txt` de dentro do `.zip` nacional, em streaming (o arquivo tem ~3,27 GB descompactado e 12,6 milhões de linhas — não cabe inteiro na memória).
2. **Transform** — filtra apenas os estabelecimentos do município de São José dos Campos (código IBGE **354990**), limpa os campos (a base vem com espaçamento fixo) e segmenta os dados de acordo com os problemas já mapeados no Log de Decisões de Dados:
   - **DEC-001**: separa estabelecimentos com RAIS negativa (zero vínculos) dos ativos.
   - **DEC-002**: separa por tipo de estabelecimento (Cnpj, Caepf, Cno).
3. **Load** — grava os CSVs tratados, prontos para uso no Power BI.

> **Pendências conhecidas (não resolvidas pelo tratamento de dados):**
> - **DEC-003** — o arquivo bruto não possui nenhuma coluna de ano/competência. Aguardando confirmação do cliente.
> - **DEC-004** — o arquivo bruto não possui razão social nem CNPJ/CPF identificado. Aguardando decisão do cliente sobre fornecer base identificada.

## 0. Upload do arquivo bruto

Faça upload do arquivo `RAIS_ESTAB_PUB.zip` (baixado do portal da RAIS) quando solicitado abaixo.
Se preferir usar o Google Drive em vez do upload direto, comente a célula de upload e ajuste o caminho `INPUT_ZIP` manualmente.

In [ ]:
from google.colab import files

uploaded = files.upload()  # selecione o RAIS_ESTAB_PUB.zip
INPUT_ZIP = list(uploaded.keys())[0]
print(f"Arquivo carregado: {INPUT_ZIP}")

## 1. Configuração

In [ ]:
import zipfile
import csv
import io
import os
import json

# Código IBGE do município de São José dos Campos
SJC_CODE = "354990"

OUTPUT_DIR = "RAIS_SJC_tratado"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Índices (0-based) das colunas no arquivo bruto RAIS_ESTAB_PUB
IDX_MUNICIPIO = 14
IDX_RAIS_NEGATIVA = 12
IDX_QTD_VINC_ATIVOS = 7
IDX_TIPO_ESTAB_LABEL = 20

# Cabeçalho de saída (nomes de colunas normalizados, sem acento/espaço)
HEADER_OUT = [
    "Bairros_SP", "Bairros_Fortaleza", "Bairros_RJ", "CNAE20_Classe", "CNAE95_Classe",
    "Distritos_SP", "Qtd_Vinculos_CLT", "Qtd_Vinculos_Ativos", "Qtd_Vinculos_Estatutarios",
    "Ind_Atividade_Ano", "Ind_CEI_Vinculado", "Ind_Estab_Participa_PAT", "Ind_Rais_Negativa",
    "Ind_Simples", "Municipio", "Natureza_Juridica", "Regioes_Adm_DF", "CNAE20_Subclasse",
    "Tamanho_Estabelecimento", "Tipo_Estab_Codigo", "Tipo_Estab_Label", "UF", "IBGE_Subsetor",
    "CEP_Estab",
]

## 2. Extract + Transform + Load

O arquivo é processado **linha a linha em streaming** (sem descompactar tudo em disco e sem carregar tudo em memória), filtrando somente os registros de SJC e já gravando nos CSVs segmentados.

In [ ]:
paths = {
    "completo": os.path.join(OUTPUT_DIR, "RAIS_SJC_completo.csv"),
    "ativos": os.path.join(OUTPUT_DIR, "RAIS_SJC_ativos.csv"),
    "rais_negativa": os.path.join(OUTPUT_DIR, "RAIS_SJC_rais_negativa.csv"),
    "cnpj": os.path.join(OUTPUT_DIR, "RAIS_SJC_Cnpj.csv"),
    "caepf": os.path.join(OUTPUT_DIR, "RAIS_SJC_Caepf.csv"),
    "cno": os.path.join(OUTPUT_DIR, "RAIS_SJC_Cno.csv"),
}

files_out = {k: open(v, "w", encoding="utf-8", newline="") for k, v in paths.items()}
writers = {k: csv.writer(f) for k, f in files_out.items()}
for w in writers.values():
    w.writerow(HEADER_OUT)

stats = {
    "total_linhas_brutas": 0,
    "total_sjc": 0,
    "sjc_ativos": 0,
    "sjc_rais_negativa": 0,
    "sjc_cnpj": 0,
    "sjc_caepf": 0,
    "sjc_cno": 0,
    "sjc_outro_tipo": 0,
}

with zipfile.ZipFile(INPUT_ZIP) as zf:
    inner_name = zf.namelist()[0]
    with zf.open(inner_name) as raw:
        # A base pública da RAIS é distribuída em Latin-1 (ISO-8859-1), não em UTF-8
        text_stream = io.TextIOWrapper(raw, encoding="latin-1", newline="")
        reader = csv.reader(text_stream, delimiter=";")
        next(reader)  # descarta cabeçalho original

        for row in reader:
            stats["total_linhas_brutas"] += 1
            if len(row) < 24:
                continue

            municipio = row[IDX_MUNICIPIO].strip()
            if municipio != SJC_CODE:
                continue

            clean_row = [c.strip() for c in row]
            stats["total_sjc"] += 1
            writers["completo"].writerow(clean_row)

            # Segmentação DEC-001: RAIS negativa vs ativos
            rais_negativa = clean_row[IDX_RAIS_NEGATIVA]
            if rais_negativa == "1":
                writers["rais_negativa"].writerow(clean_row)
                stats["sjc_rais_negativa"] += 1
            else:
                writers["ativos"].writerow(clean_row)
                stats["sjc_ativos"] += 1

            # Segmentação DEC-002: tipo de estabelecimento
            tipo_label = clean_row[IDX_TIPO_ESTAB_LABEL]
            if tipo_label == "Cnpj":
                writers["cnpj"].writerow(clean_row)
                stats["sjc_cnpj"] += 1
            elif tipo_label == "Caepf":
                writers["caepf"].writerow(clean_row)
                stats["sjc_caepf"] += 1
            elif tipo_label == "Cno":
                writers["cno"].writerow(clean_row)
                stats["sjc_cno"] += 1
            else:
                stats["sjc_outro_tipo"] += 1

for f in files_out.values():
    f.close()

stats["pct_rais_negativa"] = round(100 * stats["sjc_rais_negativa"] / stats["total_sjc"], 2) if stats["total_sjc"] else 0

with open(os.path.join(OUTPUT_DIR, "estatisticas_etl.json"), "w", encoding="utf-8") as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)

print(json.dumps(stats, ensure_ascii=False, indent=2))

## 3. Conferência rápida dos resultados

In [ ]:
import pandas as pd

df_completo = pd.read_csv(os.path.join(OUTPUT_DIR, "RAIS_SJC_completo.csv"))
print(f"Total de estabelecimentos em SJC: {len(df_completo)}")
df_completo.head()

## 4. Download dos CSVs tratados

Baixe os arquivos para subir na pasta `notebooks/` (ou `data/`) do repositório GitHub e usar como fonte no Power BI.

In [ ]:
import shutil

shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
files.download(f"{OUTPUT_DIR}.zip")

## 5. Pendências para a próxima sprint

- **DEC-003** — confirmar com o cliente o ano/competência de referência deste arquivo (não presente nos dados).
- **DEC-004** — aguardar decisão do cliente sobre fornecimento de base identificada (razão social/CNPJ).
- Implementar geocodificação a partir do campo `CEP_Estab` para viabilizar o mapeamento geográfico (User Story Rank 2).